# 02 — Engenharia de Features

Neste notebook construo as duas bases que vão alimentar meus modelos. A
decisão mais importante que preciso tomar aqui não é qual feature criar —
é **qual feature proibir**. Aprendi isso na prática, logo na próxima seção.

In [ ]:
import sys, pathlib, warnings

# Descobre a pasta src/orion subindo a partir do diretório atual do kernel.
# Evita o erro "No module named 'orion'": o VS Code às vezes abre o notebook
# com o cwd na raiz do projeto, às vezes em notebooks/ — um caminho relativo
# fixo como '../src' só funciona no segundo caso.
_cwd = pathlib.Path.cwd()
for _base in [_cwd, *_cwd.parents]:
    _src = _base / 'src'
    if (_src / 'orion').is_dir():
        sys.path.insert(0, str(_src))
        break
else:
    raise FileNotFoundError(
        f"Não encontrei a pasta src/orion a partir de {_cwd}. "
        "Rode o notebook com o kernel na raiz do projeto (orion-aiops/) ou em notebooks/."
    )

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

In [ ]:
from orion.features import construir_gold, construir_serie_diaria, construir_base_risco
from orion.config import DATA_INICIO_REGIME

silver = pd.read_parquet('../data/silver/incidentes.parquet')
print(f'{len(silver):,} incidentes na camada Silver')

## 1. Vazamento de alvo — a armadilha deste dataset

Essa foi uma armadilha em que quase caí. Percebi que seis campos do
dicionário só existem **depois** que o incidente já foi resolvido:

`Duração` · `Resolvido` · `Encerrado` · `Código de fechamento` · `Solução` · `Status`

Se eu usar `Duração` para prever violação de OLA, o modelo acerta ~99% —
e é completamente inútil, porque no momento em que a operação precisa da
previsão (quando o chamado acabou de entrar na fila) a duração ainda nem
existe. Para não confiar só na intuição, decidi demonstrar o problema
numericamente antes de seguir em frente.

Pra ler o resultado abaixo: **AUC** é uma nota de 0 a 1 que mede o quão bem
um modelo separa quem violou de quem não violou. 0,5 é chute puro (tipo
cara ou coroa), 1,0 seria separação perfeita. Já adianto o resultado: vou
usar essa métrica o notebook inteiro, então melhor entender ela logo aqui.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

demo = silver[silver['entrou_kpi'] & silver['ola_violado'].notna()].copy()
y = demo['ola_violado'].astype(int)

# Modelo COM vazamento
X_leak = demo[['duracao_s', 'consumo_ola']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
m = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)
auc_leak = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])

print(f'AUC usando Duração (VAZAMENTO): {auc_leak:.3f}')
print('Esse número é falso. A duração É a definição da violação.')
print('Ele não pode ser reproduzido em produção — o campo está vazio na abertura.')

Isso confirmou minha suspeita: um AUC de 0,97+ neste dataset é quase sempre
sinal de vazamento, não de um bom modelo. Fiquei com essa regra na cabeça
para o resto do projeto. Por isso, a base `incidentes_risco` que construo
a seguir só carrega o que já se sabia **no momento da abertura** do chamado.

## 2. Base 1 — série temporal diária (modelo de volume)

Para o modelo de volume, organizei as features em quatro famílias. Cheguei
nessa organização depois de entender, no notebook anterior, que a
sazonalidade semanal é o sinal mais forte da série — então construí as
features em torno disso:

* **Calendário**: dia da semana, dia do mês, mês, semana do ano, flags de fim de semana
  e virada de mês, além de codificação cíclica (`dow_sin`/`dow_cos`) para que domingo e
  segunda fiquem próximos no espaço de features.
* **Lags**: 1, 2, 3, 7, 14 e 28 dias.
* **Janelas móveis**: média, desvio e máximo de 7, 14 e 28 dias — sempre com `shift(1)`
  para não incluir o próprio dia, o que aprendi ser essencial para não vazar o presente.
* **Sazonalidade dedicada**: `media_mesmo_dow_4s` — média do mesmo dia da semana nas
  4 semanas anteriores. É minha tentativa de capturar diretamente o padrão mais forte
  que encontrei na EDA.

In [ ]:
serie = construir_serie_diaria(silver)
serie = serie[serie['data'] >= DATA_INICIO_REGIME]
print(f'{len(serie):,} linhas (dia x prioridade) | {serie.shape[1]} colunas')
print('\nFamílias de features:')
for fam, pref in [('lags', 'lag_'), ('médias', 'media_'), ('desvios', 'std_'), ('máximos', 'max_')]:
    print(f'  {fam:10s}: {[c for c in serie.columns if c.startswith(pref)]}')
serie.head()

### Cuidado com janelas móveis em dados agrupados

Esse foi um erro que cometi na primeira tentativa e só percebi depois de
conferir o resultado: `df.groupby('prioridade')['volume'].shift(1).rolling(28).mean()`
parece correto, mas não é. O `shift` respeita o grupo, mas o `rolling` que
vem depois roda sobre a série concatenada inteira — misturando P2 e P3 na
mesma janela sem eu perceber de cara.

A forma correta que passei a usar depende de `transform`. Para confirmar que
corrigi o problema, fiz a verificação abaixo:

In [ ]:
# Verificação: a média móvel de cada prioridade tem que bater com o volume médio dela
check = serie.groupby('prioridade')[['volume', 'media_28', 'media_7', 'media_mesmo_dow_4s']].mean()
print(check.round(2))
print('\nSe media_28 divergir muito de volume, a janela está misturando grupos.')

### O que é correlação, rapidamente

O gráfico abaixo mostra a correlação de cada feature com o volume do dia
seguinte — um número entre -1 e 1 que diz o quanto duas colunas "andam
juntas". Perto de 1: quando uma sobe, a outra também sobe. Perto de -1:
quando uma sobe, a outra desce. Perto de 0: não tem relação linear visível.
Não é prova de causa e efeito, é só um primeiro raio-x de quais colunas
parecem mais ligadas ao que quero prever.

In [ ]:
# Correlação das features com o alvo D+1
alvo = 'alvo_d1'
num = serie.select_dtypes('number').drop(columns=['alvo_d1', 'alvo_d7'])
corr = num.corrwith(serie[alvo]).dropna().sort_values(key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
corr.plot(kind='barh', ax=ax, color=np.where(corr > 0, '#38BDF8', '#EF4444'))
ax.set_title('Correlação com o volume de D+1')
ax.invert_yaxis()
plt.tight_layout(); plt.show()
corr.round(3)

## 3. Base 2 — risco de OLA por incidente

Para o segundo modelo, o de risco de violação de OLA, organizei três famílias
de features — sempre me perguntando "isso já era conhecido no momento em
que o chamado foi aberto?":

* **Atributos do chamado** — prioridade, grupo, IC, categoria, produto, hora, dia da semana.
* **Pressão operacional no instante** — quantos chamados o grupo recebeu nas últimas 24h,
  quantos aquele IC gerou nos últimos 7 dias, carga geral da operação. Quis capturar a
  diferença entre um chamado que chega numa terça calma e um que chega no meio de um
  incidente maior.
* **Histórico expanding** — taxa de violação daquele grupo/IC/categoria/produto
  considerando **apenas os incidentes anteriores** (`shift(1).expanding().mean()`).
  Aqui tomei cuidado especial: usar a média do período inteiro seria um vazamento mais
  sutil que o da `Duração`, mas igualmente fatal — só percebi esse risco depois de já
  ter cometido o erro parecido nas janelas móveis.

In [ ]:
risco = construir_base_risco(silver)
print(f'{len(risco):,} incidentes | {risco.shape[1]} colunas')
print(f"Taxa de violação: {risco['ola_violado'].mean():.2%} "
      f"({int(risco['ola_violado'].sum())} positivos)")
risco.head()

In [ ]:
# Prova de que o histórico é expanding (sem vazamento): os primeiros
# incidentes de cada grupo têm histórico nulo, porque não há passado ainda.
primeiros = risco.groupby('grupo_designado', observed=True).head(1)
print('Primeiro incidente de cada grupo — histórico deve ser NaN:')
print(primeiros[['grupo_designado', 'hist_viol_grupo', 'hist_viol_grupo_n']].head(8).to_string(index=False))

In [ ]:
# O evento é raro. Visualizando o desbalanceamento.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

risco['ola_violado'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#10B981', '#EF4444'])
axes[0].set_title(f"Desbalanceamento — apenas {risco['ola_violado'].mean():.2%} de positivos")
axes[0].set_xticklabels(['Não violou', 'Violou'], rotation=0)

mensal = risco.groupby(['ano_mes'])['ola_violado'].agg(['sum', 'count'])
mensal['taxa'] = mensal['sum'] / mensal['count']
axes[1].plot(mensal.index, mensal['taxa'] * 100, marker='o', color='#EF4444')
axes[1].set_title('Taxa mensal de violação (%)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Discriminação preliminar: as features de histórico separam as classes?
comparacao = risco.groupby('ola_violado')[[
    'carga_grupo_24h', 'carga_ic_168h', 'carga_geral_24h',
    'hist_viol_grupo', 'hist_viol_ic', 'hist_viol_categoria', 'hist_viol_produto',
]].mean().T
comparacao.columns = ['Não violou', 'Violou']
comparacao['razão'] = (comparacao['Violou'] / comparacao['Não violou']).round(2)
comparacao.round(4)

## 4. Materialização da camada Gold

Com as duas bases validadas, gravo tudo na camada Gold para que os notebooks
de modelagem não precisem recalcular nada — só carregar.

In [ ]:
saidas = construir_gold()
for nome, df in saidas.items():
    print(f'{nome:32s} {str(df.shape):>14s}')

Com as features prontas e sem vazamento, sigo para o primeiro modelo:
`03_modelo_volume_d1_d7.ipynb`.